# EDA | Observatório de mercado B3

**Autor: Rodrigo Correa**

**Linha 2 · Período: 01/01/2021–31/12/2023 · 12 ativos · 4 grupos didáticos**

Perguntas: quais ativos tiveram maior retorno? Como retorno e risco se relacionam?
Qual foi a maior perda desde um topo? Os retornos apresentam caudas e correlação?

Fonte: snapshot Yahoo Finance via yfinance. `EMBR3` mantém o código histórico;
seu histórico foi adquirido pelo alias `EMBJ3.SA`, conforme manifesto.
Volume Financeiro é uma estimativa `Close × Volume`, não giro oficial da B3.

Execute todas as células com o ambiente de `requirements-dev.txt`.
A análise é retrospectiva e não considera custos ou impostos.

In [1]:
import hashlib
import json
import logging
import os
import sys
from pathlib import Path

# Aceita abertura na raiz ou em notebooks/, usando somente caminhos relativos.
if not Path("data").is_dir() and (Path("..") / "data").is_dir():
    os.chdir("..")
if not Path("data").is_dir():
    raise RuntimeError("Abra este notebook na raiz do projeto ou em notebooks/.")
sys.path.insert(0, ".")
for name in [
    "streamlit.runtime.caching.cache_data_api",
    "streamlit.runtime.scriptrunner_utils.script_run_context",
]:
    logging.getLogger(name).setLevel(logging.ERROR)

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

from src import charts
from src.analytics import analyze, distribution_stats, enrich_data, return_correlations
from src.config import DATA_PATH, MANIFEST_PATH, TICKERS
from src.schema import validate_data

pd.set_option("display.max_columns", 15)
px.defaults.template = "plotly_dark"
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == manifest["sha256"]
df = validate_data(pd.read_csv(DATA_PATH, parse_dates=["Data"]))
print(f"Fonte: {manifest['source']} | Linhas: {len(df):,} | SHA256 conferido")
print(f"Pregões: {df['Data'].min():%d/%m/%Y} a {df['Data'].max():%d/%m/%Y}")
display(df.head())

2026-10-31 13:01:21.090 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-31 13:01:21.092 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-31 13:01:21.093 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-31 13:01:21.095 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-31 13:01:21.096 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


...

2026-10-31 13:01:21.099 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-31 13:01:21.121 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-31 13:01:21.149 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Fonte: Yahoo Finance via yfinance | Linhas: 8,940 | SHA256 conferido
Pregões: 04/01/2021 a 28/12/2023


,Data,Ativo,Setor,Open,High,Low,Close,Adj Close,Volume,Volume Financeiro
0,2021-01-04,ABEV3,Consumo e serviços,15.790000,15.850000,15.380000,15.480000,11.716109,17835200,2.760889e+08
1,2021-01-04,BBDC4,Financeiro,22.752066,22.752066,21.702478,21.842976,14.483535,48196357,1.052752e+09
2,2021-01-04,EGIE3,Energia,31.357141,31.907143,30.821428,31.128571,22.430519,3114020,9.693499e+07
3,2021-01-04,EMBR3,Indústria e materiais,8.940000,9.000000,8.310000,8.370000,8.244199,17114300,1.432467e+08
4,2021-01-04,GGBR4,Indústria e materiais,19.682539,20.666666,19.682539,20.666666,13.697057,27916686,5.769448e+08


## 1. Qualidade e cobertura

O validador rejeita nulos, duplicatas, valores inválidos, datas fora do contrato,
universo incompleto, falhas de cobertura anual e calendários divergentes.
Não é feito preenchimento de preços. A validação de cobertura não compara a base
com um calendário oficial de negociação independente.

In [3]:
assert set(df["Ativo"]) == set(TICKERS)
assert df.isna().sum().sum() == 0
assert not df.duplicated(["Data", "Ativo"]).any()
coverage = df.assign(Ano=df["Data"].dt.year).groupby(["Ativo", "Ano"]).size().unstack()
display(coverage)
display(df.dtypes.astype(str).rename("Tipo").to_frame())
print(
    "Nulos:",
    int(df.isna().sum().sum()),
    "| Duplicatas:",
    int(df.duplicated(["Data", "Ativo"]).sum()),
)

Ano,2021,2022,2023
Ativo,,,
ABEV3,247,250,248
BBDC4,247,250,248
EGIE3,247,250,248
EMBR3,247,250,248
GGBR4,247,250,248
ITUB4,247,250,248
KLBN11,247,250,248
PETR4,247,250,248
PRIO3,247,250,248


,Tipo
Data,datetime64[us]
Ativo,str
Setor,str
Open,float64
High,float64
Low,float64
Close,float64
Adj Close,float64
Volume,int64
Volume Financeiro,float64


Nulos: 0 | Duplicatas: 0


## 2. Retornos e estatísticas

Preços ajustados permitem comparar séries com eventos corporativos considerados
pelo provedor. Retorno simples: `P(t)/P(t-1) - 1`. O primeiro dia é excluído da
amostra de retornos. Volatilidade: desvio padrão amostral × √252.

In [4]:
enriched = enrich_data(df)
long, metrics, portfolio = analyze(enriched)
ranking = metrics.sort_values("Retorno Acumulado %", ascending=False).reset_index(drop=True)
display(ranking.round(3))
display(distribution_stats(long).round(3))
display(metrics.sort_values("Retorno / risco acumulado", ascending=False).round(3))
charts.base100(long).show()

2026-10-31 13:01:21.298 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-31 13:01:21.305 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-10-31 13:01:21.326 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-31 13:01:21.334 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


,Ativo,Retorno Acumulado %,Volatilidade Anualizada %,Retorno / risco acumulado,Sharpe empírico (rf=0),Drawdown Máximo %,Volume Financeiro Médio,Setor
0,PETR4,228.689,39.422,5.801,1.223,-39.104,2.098762e+09,Energia
1,PRIO3,208.315,46.781,4.453,1.048,-34.316,4.510331e+08,Energia
2,EMBR3,167.503,46.821,3.578,0.944,-58.273,1.608476e+08,Indústria e materiais
3,GGBR4,30.795,35.556,0.866,0.433,-33.931,3.448578e+08,Indústria e materiais
4,EGIE3,25.347,18.762,1.351,0.502,-17.652,6.539772e+07,Energia
5,ITUB4,24.025,28.834,0.833,0.399,-35.869,9.371190e+08,Financeiro
6,VALE3,18.109,33.900,0.534,0.335,-41.343,2.321986e+09,Indústria e materiais
7,WEGE3,3.505,32.827,0.107,0.199,-49.510,2.930961e+08,Indústria e materiais
8,ABEV3,2.570,25.465,0.101,0.160,-31.416,3.891388e+08,Consumo e serviços
9,RENT3,-0.250,36.889,-0.007,0.182,-38.399,4.017009e+08,Consumo e serviços


2026-10-31 13:01:21.391 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-31 13:01:21.396 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


,Ativo,Observações,Média diária %,Desvio diário %,Assimetria,Excesso de curtose,Percentil 1%,Percentil 5%,Mediana %,Percentil 95%,Percentil 99%,Média da cauda 5%,Pior dia %
0,ABEV3,744,0.016,1.604,0.531,3.461,-3.863,-2.493,0.000,2.681,4.361,-3.320,-5.701
1,BBDC4,744,0.010,1.977,-0.991,8.992,-4.703,-2.970,0.000,3.086,4.958,-4.505,-17.384
2,EGIE3,744,0.037,1.182,0.029,1.613,-3.047,-1.803,0.026,1.990,2.781,-2.561,-4.751
3,EMBR3,744,0.175,2.949,0.589,4.242,-6.856,-4.395,0.000,4.876,8.659,-6.010,-14.927
4,GGBR4,744,0.061,2.240,0.105,0.792,-5.252,-3.711,0.036,3.879,5.805,-4.687,-7.932
5,ITUB4,744,0.046,1.816,-1.115,13.494,-3.935,-2.477,0.000,2.893,4.036,-3.872,-17.964
6,KLBN11,744,0.013,1.677,0.288,1.278,-3.781,-2.602,-0.038,2.724,4.623,-3.351,-6.247
7,PETR4,744,0.191,2.483,-0.866,7.629,-6.538,-3.881,0.169,3.985,6.096,-5.728,-20.710
8,PRIO3,744,0.195,2.947,0.303,2.188,-6.929,-4.371,0.146,5.009,7.693,-5.934,-10.763
9,RENT3,744,0.027,2.324,0.178,1.293,-5.683,-3.613,-0.072,3.956,6.032,-4.823,-9.387


,Ativo,Retorno Acumulado %,Volatilidade Anualizada %,Retorno / risco acumulado,Sharpe empírico (rf=0),Drawdown Máximo %,Volume Financeiro Médio,Setor
7,PETR4,228.689,39.422,5.801,1.223,-39.104,2.098762e+09,Energia
8,PRIO3,208.315,46.781,4.453,1.048,-34.316,4.510331e+08,Energia
3,EMBR3,167.503,46.821,3.578,0.944,-58.273,1.608476e+08,Indústria e materiais
2,EGIE3,25.347,18.762,1.351,0.502,-17.652,6.539772e+07,Energia
4,GGBR4,30.795,35.556,0.866,0.433,-33.931,3.448578e+08,Indústria e materiais
5,ITUB4,24.025,28.834,0.833,0.399,-35.869,9.371190e+08,Financeiro
10,VALE3,18.109,33.900,0.534,0.335,-41.343,2.321986e+09,Indústria e materiais
11,WEGE3,3.505,32.827,0.107,0.199,-49.510,2.930961e+08,Indústria e materiais
0,ABEV3,2.570,25.465,0.101,0.160,-31.416,3.891388e+08,Consumo e serviços
9,RENT3,-0.250,36.889,-0.007,0.182,-38.399,4.017009e+08,Consumo e serviços


## 3. Relação entre risco, retorno e liquidez

A razão retorno acumulado/volatilidade anualizada responde à comparação pedida
no enunciado; não é o Sharpe clássico. O Sharpe empírico usa média diária /
desvio diário × raiz de 252, com taxa livre de risco explicitamente zero.
Não representa retorno excedente ao CDI/Selic. Razões com risco nulo ficam indisponíveis.

Cada bolha usa volume financeiro médio estimado. Os horizontes do retorno acumulado
e da volatilidade anualizada são diferentes e estão identificados. Assimetria,
curtose de Fisher e quantis descrevem a amostra, sem pressupor normalidade.


In [5]:
charts.risk_return(metrics).show()
charts.histogram(long).show()

## 4. Drawdown e tendência

O drawdown é reiniciado em 04/01/2021 e usa o topo conhecido até cada pregão.
As médias móveis também usam somente passado e presente; janelas incompletas
no início do histórico permanecem nulas.

In [6]:
charts.drawdown(long).show()
charts.moving_averages(enriched, "WEGE3").show()

## 5. Correlação dos retornos diários

Correlação de Pearson sobre observações alinhadas. Ela mede associação linear
histórica, não causalidade ou garantia de diversificação futura.

In [7]:
daily = long.pivot(index="Data", columns="Ativo", values="Retorno Diário %").dropna()
charts.correlations(return_correlations(long)).show()
print("Observações comuns para a correlação:", len(daily))


2026-10-31 13:01:22.772 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-31 13:01:22.778 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Observações comuns para a correlação: 744


## 6. Síntese reproduzível

Os resultados abaixo são calculados diretamente do snapshot, sem números fixados
no texto. A comparação usa pesos iguais no início e não faz rebalanceamento.

In [8]:
best = ranking.iloc[0]
worst = ranking.iloc[-1]
worst_dd = metrics.loc[metrics["Drawdown Máximo %"].idxmin()]
resilient = metrics.loc[metrics["Drawdown Máximo %"].idxmax()]
risk_leader = metrics.loc[metrics["Retorno / risco acumulado"].idxmax()]
portfolio_return = float(portfolio.iloc[-1] - 100)
assert np.isclose(portfolio_return, metrics["Retorno Acumulado %"].mean())
display(
    Markdown(
        f"- Maior retorno acumulado: **{best['Ativo']} ({best['Retorno Acumulado %']:.2f}%)**.\n"
        f"- Menor retorno acumulado: **{worst['Ativo']} ({worst['Retorno Acumulado %']:.2f}%)**.\n"
        f"- Retorno da carteira: **{portfolio_return:.2f}%**.\n"
        f"- Maior drawdown em magnitude: **{worst_dd['Ativo']} "
        f"({worst_dd['Drawdown Máximo %']:.2f}%)**.\n"
        f"- Volatilidade média dos ativos: **"
        f"{metrics['Volatilidade Anualizada %'].mean():.2f}% a.a.**."
    )
)

- Maior retorno acumulado: **PETR4 (228.69%)**.
- Menor retorno acumulado: **BBDC4 (-6.93%)**.
- Retorno da carteira: **58.41%**.
- Maior drawdown em magnitude: **EMBR3 (-58.27%)**.
- Volatilidade média dos ativos: **33.61% a.a.**.

In [9]:
print("Menor drawdown:", resilient["Ativo"], round(resilient["Drawdown Máximo %"], 2))
print("Melhor retorno/risco:", risk_leader["Ativo"], round(risk_leader["Retorno / risco acumulado"], 2))


Menor drawdown: EGIE3 -17.65
Melhor retorno/risco: PETR4 5.8


## Limitações e referências

- Universo fixado pelo enunciado: possível viés de seleção.
- Quatro grupos didáticos; KLBN11 é uma unit.
- Retornos dependem dos ajustes históricos do provedor e podem ser revisados.
- `Close × Volume` não substitui volume financeiro oficial da B3.
- Não há custos, tributos ou simulação de execução real de ordens.
- A volatilidade média individual não equivale à volatilidade da carteira.

[yfinance.download](https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html)
· [Embraer RI](https://ri.embraer.com.br/governanca/composicao-acionaria-2/)
· Metodologia detalhada: `README.md` e aba Metodologia do dashboard.